# Lab 4 – Kafka overview

**BRICOUT – QUARANTA**

Subject: [04.kafka_overview/lab.md](https://github.com/adaltas/ece-big-data-processing-fall-2026/blob/main/04.kafka_overview/lab.md)

1. Setup: `confluent-kafka` and a Kafka broker in Docker
2. Original demo: topic `timer` (admin / producer / consumer)
3. Lab: topic `book-lines` (Pride and Prejudice, Project Gutenberg) and text cleaning into `cleaned.txt`
4. Cleanup

## 1. Setup

In [1]:
%pip install -q confluent-kafka

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
!docker pull apache/kafka-native:4.1.1
!docker run -d --name kafka-lab -p 9092:9092 apache/kafka-native:4.1.1

4.1.1: Pulling from apache/kafka-native
Digest: sha256:b8faca36f5385f90e46861ea0b2f532ebed86e9b2868b717c75a67f6fda0c7ce
Status: Image is up to date for apache/kafka-native:4.1.1
docker.io/apache/kafka-native:4.1.1


fdff479bec30ebdce22e13a7b654ae1345de593552e0cbe7f8fb280da0a58b74


In [3]:
# Wait for the broker to be ready
import time
time.sleep(10)

## 2. Original demo – topic `timer`

### Admin: create the topic

In [4]:
from confluent_kafka.admin import AdminClient, NewTopic

config =  {
  'bootstrap.servers': 'localhost:9092',
}

admin_client = AdminClient(config)

topic='timer'
futures = admin_client.create_topics(
  [NewTopic(topic, num_partitions=1, replication_factor=1)]
)
for t, f in futures.items():
  f.result()
  print(f"Topic '{t}' created")

x = admin_client.list_topics()
for  t in x.topics.keys():
  print(t)

Topic 'timer' created
timer


### Producer: send the current time every second

The original script streams for 5 minutes; the window is reduced to 15 seconds here.

In [5]:
import socket
import time
from confluent_kafka import Producer
from datetime import datetime, timedelta

conf = {'bootstrap.servers': 'localhost:9092',
        'client.id': socket.gethostname()}

producer = Producer(conf)

topic='timer'
message='The time is now '

# Streaming Query
duration = 0.25 # Streaming window in minutes (5 in the original script)
start_time = datetime.now() # Current clock time
stop_time = start_time + timedelta(minutes=duration) #start+duration=stop

while datetime.now() < stop_time:
  time_now = datetime.now().strftime("%H:%M:%S")
  producer.produce(
    topic=topic,
    value=message + time_now
  )
  print(message + time_now)
  time.sleep(1)

producer.flush()

The time is now 17:03:23


The time is now 17:03:24


The time is now 17:03:25


The time is now 17:03:26


The time is now 17:03:27


...

The time is now 17:03:34


The time is now 17:03:35


The time is now 17:03:36


The time is now 17:03:37


0

### Consumer: read the topic, stop after ~10 seconds of silence

In [6]:
from confluent_kafka import Consumer

conf = {'bootstrap.servers': 'localhost:9092',
        'group.id': 'foo',
        'auto.offset.reset': 'smallest'}

consumer = Consumer(conf)

topic='timer'
consumer.subscribe([topic])

# Configuration
MAX_EMPTY_POLLS = 10  # Ends after ~10 seconds of silence
MAX_ERRORS = 5        # Ends after 5 consecutive errors
empty_polls = 0
error_count = 0

while True:
    msg = consumer.poll(1.0)

    # 1. Handle "No Message" (Timeout)
    if msg is None:
        empty_polls += 1
        if empty_polls >= MAX_EMPTY_POLLS:
            print("Closing: No new messages received.")
            break
        continue

    # 2. Handle Errors
    if msg.error():
        error_count += 1
        print(f"Consumer error: {msg.error()}")
        if error_count >= MAX_ERRORS:
            print("Closing: Too many consecutive errors.")
            break
        continue

    # 3. Handle Success
    # Reset counters when we actually get data
    empty_polls = 0
    error_count = 0

    print(f"Rcvd message: {msg.value().decode('utf-8')}")

# Clean up
consumer.close()

Rcvd message: The time is now 17:03:23
Rcvd message: The time is now 17:03:24
Rcvd message: The time is now 17:03:25
Rcvd message: The time is now 17:03:26
Rcvd message: The time is now 17:03:27
Rcvd message: The time is now 17:03:28
Rcvd message: The time is now 17:03:29
Rcvd message: The time is now 17:03:30
Rcvd message: The time is now 17:03:31
Rcvd message: The time is now 17:03:32
Rcvd message: The time is now 17:03:33
Rcvd message: The time is now 17:03:34
Rcvd message: The time is now 17:03:35
Rcvd message: The time is now 17:03:36
Rcvd message: The time is now 17:03:37


Closing: No new messages received.


## 3. Lab – topic `book-lines`

### Download the book

Pride and Prejudice, Project Gutenberg, saved as `book.txt`.

In [7]:
import os
import urllib.request

book_path='book.txt'
if not os.path.exists(book_path):
  urllib.request.urlretrieve('https://www.gutenberg.org/cache/epub/1342/pg1342.txt', book_path)
print(f"{book_path}: {os.path.getsize(book_path)} bytes")

book.txt: 772386 bytes


### Admin: create the topic `book-lines`

In [8]:
from confluent_kafka.admin import AdminClient, NewTopic

config =  {
  'bootstrap.servers': 'localhost:9092',
}

admin_client = AdminClient(config)

topic='book-lines'
futures = admin_client.create_topics(
  [NewTopic(topic, num_partitions=1, replication_factor=1)]
)

# Wait for the creation result (an error is raised if the topic already exists)
for t, f in futures.items():
  try:
    f.result()
    print(f"Topic '{t}' created")
  except Exception as e:
    print(f"Topic '{t}' not created: {e}")

x = admin_client.list_topics()
for  t in x.topics.keys():
  print(t)

Topic 'book-lines' created
timer
__consumer_offsets
book-lines


### Producer: send every non-empty line of the book

In [9]:
import socket
from confluent_kafka import Producer

conf = {'bootstrap.servers': 'localhost:9092',
        'client.id': socket.gethostname()}

producer = Producer(conf)

topic='book-lines'
book_path='book.txt'

# Send every non-empty line of the book to the topic
sent = 0
with open(book_path, encoding='utf-8-sig') as book:
  for line in book:
    line = line.strip()
    if not line:
      continue
    producer.produce(topic=topic, value=line.encode('utf-8'))
    producer.poll(0)  # serve delivery callbacks / free the local queue
    sent += 1

producer.flush()
print(f"{sent} lines sent to topic '{topic}'")

11882 lines sent to topic 'book-lines'


### Consumer: read from the beginning, clean and write `cleaned.txt`

Cleaning, inspired by the word count lab:

1. lower case
2. punctuation and digits removed (`[^a-z\s]` replaced by a space)
3. tokenisation into words (`split()`, which also drops blank spaces)

There is one output line per message, with words separated by a single space. Lines left empty after cleaning (e.g. `***` or chapter numbers) are not written. The consumer stops after 10 seconds without a new message.

In [10]:
import re
from confluent_kafka import Consumer

conf = {'bootstrap.servers': 'localhost:9092',
        'group.id': 'book-cleaner',
        'auto.offset.reset': 'earliest'}

consumer = Consumer(conf)

topic='book-lines'
consumer.subscribe([topic])
output_path='cleaned.txt'

# Text cleaning (inspired by the word count lab)
def clean(line):
  line = line.lower()                    # b. lower case
  line = re.sub(r'[^a-z\s]', ' ', line)  # f. remove punctuation and digits
  return line.split()                    # tokenize, drops blank spaces

# Configuration
MAX_EMPTY_POLLS = 10  # Ends after ~10 seconds of silence
MAX_ERRORS = 5        # Ends after 5 consecutive errors
empty_polls = 0
error_count = 0
received = 0

with open(output_path, 'w', encoding='utf-8') as out:
  while True:
    msg = consumer.poll(1.0)

    # 1. Handle "No Message" (Timeout)
    if msg is None:
      empty_polls += 1
      if empty_polls >= MAX_EMPTY_POLLS:
        print("Closing: No new messages received.")
        break
      continue

    # 2. Handle Errors
    if msg.error():
      error_count += 1
      print(f"Consumer error: {msg.error()}")
      if error_count >= MAX_ERRORS:
        print("Closing: Too many consecutive errors.")
        break
      continue

    # 3. Handle Success
    empty_polls = 0
    error_count = 0
    received += 1

    words = clean(msg.value().decode('utf-8'))
    if words:
      out.write(' '.join(words) + '\n')

# Clean up
consumer.close()
print(f"{received} messages processed, result written to {output_path}")

Closing: No new messages received.
11882 messages processed, result written to cleaned.txt


### Result: first 10 lines of `cleaned.txt`

In [11]:
with open('cleaned.txt', encoding='utf-8') as f:
  lines = f.read().splitlines()

print(f"cleaned.txt: {len(lines)} lines\n")
for line in lines[:10]:
  print(line)

cleaned.txt: 11847 lines

the project gutenberg ebook of pride and prejudice
this ebook is for the use of anyone anywhere in the united states and
most other parts of the world at no cost and with almost no restrictions
whatsoever you may copy it give it away or re use it under the terms
of the project gutenberg license included with this ebook or online
at www gutenberg org if you are not located in the united states
you will have to check the laws of the country where you are located
before using this ebook
title pride and prejudice
author jane austen


## 4. Cleanup: stop and remove the Kafka container

In [12]:
!docker stop kafka-lab
!docker rm kafka-lab

kafka-lab


kafka-lab
